# 📌 Grounding and Citations

## The Problem

Even with RAG, an LLM can still mix retrieved facts with things it made up.  
**Grounding** means forcing the LLM to only say things supported by the retrieved documents.  
**Citations** tell the user *which* document each claim came from.

```
Without grounding:
  Answer: "RAG uses BM25 for retrieval and GPT-4 for generation."  ← not from any doc!

With grounding + citations:
  Answer: "RAG combines a retriever with an LLM [1].
           The retriever uses embedding similarity [2]."
  [1] Source: doc_id=3, page 5
  [2] Source: doc_id=7, page 2
```

## What You'll Learn

1. Prompt tricks to enforce grounding
2. Getting the LLM to cite inline ([1], [2])
3. Parsing citations from LLM output
4. Faithfulness check — did the LLM stick to the sources?

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


In [ ]:
import re

# Simulated documents with source metadata
docs = [
    {"id": 1, "text": "RAG stands for Retrieval-Augmented Generation.",
     "source": "arxiv:2005.11401", "page": 1},

    {"id": 2, "text": "The retriever in RAG encodes documents and queries as embeddings and uses cosine similarity to rank results.",
     "source": "arxiv:2005.11401", "page": 3},

    {"id": 3, "text": "Retrieved documents are concatenated and passed as context to the generator (LLM).",
     "source": "arxiv:2005.11401", "page": 4},

    {"id": 4, "text": "RAG reduces hallucination by grounding responses in retrieved external knowledge.",
     "source": "blog.langchain.dev", "page": None},
]

question = "How does RAG work and why does it help with hallucination?"

print(f"Question: {question}\n")
for d in docs:
    print(f"  [{d['id']}] ({d['source']}) {d['text']}")

## 1. Grounding Prompt

In [ ]:
# Build a numbered context block
context_block = ""
for d in docs:
    context_block += f"[{d['id']}] {d['text']}\n"

# Grounding prompt — key lines in bold in comments
grounding_prompt = f"""You are a precise assistant. Answer using ONLY the numbered sources below.

Rules:
- Every sentence in your answer MUST be supported by a source.
- Cite each source inline like: [1] or [2].
- If a fact is not in any source, do NOT include it.
- End your answer with a "Sources used:" list.

Sources:
{context_block}
Question: {question}
Answer:"""

print(grounding_prompt)

In [ ]:
answer = llm(prompt, max_tokens=300)


Sources used: [1], [2], [3], [4]"""

print("Grounded answer with citations:")
print("-" * 50)
print(answer)

## 2. Parse Citations from the Answer

In [ ]:
# Extract which sources the LLM cited

def extract_citations(answer_text):
    """Find all [N] citation patterns in the answer."""
    # Regex: find all [number] patterns
    matches = re.findall(r'\[(\d+)\]', answer_text)
    cited_ids = sorted(set(int(m) for m in matches))
    return cited_ids


cited = extract_citations(answer)
print(f"LLM cited sources: {cited}")

# Show what those sources were
print("\nCited source details:")
doc_map = {d['id']: d for d in docs}
for cid in cited:
    d = doc_map.get(cid)
    if d:
        page_info = f"page {d['page']}" if d['page'] else "(no page)"
        print(f"  [{cid}] {d['source']} {page_info}: {d['text'][:60]}...")

## 3. Faithfulness Check

Did the LLM only use information from the sources? Let's do a simple check.

In [ ]:
# Simple faithfulness check:
# For each sentence in the answer, is there a citation?
# Sentences without citations are potential hallucinations.

def check_faithfulness(answer_text):
    """
    Split answer into sentences and flag any that
    don't contain a citation.
    """
    # Split into sentences (rough: split on '. ')
    sentences = [s.strip() for s in answer_text.split('. ') if s.strip()]

    results = []
    for sent in sentences:
        has_citation = bool(re.search(r'\[\d+\]', sent))
        results.append({
            "sentence": sent,
            "has_citation": has_citation
        })
    return results


faithfulness = check_faithfulness(answer)

print("Faithfulness check (sentence by sentence):\n")
for r in faithfulness:
    status = "✅ cited" if r["has_citation"] else "⚠️  no citation"
    print(f"  {status}: {r['sentence'][:70]}")

cited_count = sum(1 for r in faithfulness if r["has_citation"])
print(f"\nFaithfulness score: {cited_count}/{len(faithfulness)} sentences have citations")

In [ ]:
# Test with a bad answer that has unsupported claims
bad_answer = """RAG stands for Retrieval-Augmented Generation [1]. \
It was invented at Facebook in 2020 and uses BM25 for retrieval. \
The LLM generates answers based on the retrieved context [3]."""

print("Bad answer (contains unsupported claims):\n")
print(bad_answer)

print("\nFaithfulness check:")
for r in check_faithfulness(bad_answer):
    status = "✅ cited" if r["has_citation"] else "⚠️  potential hallucination!"
    print(f"  {status}: {r['sentence'][:70]}")

## 4. Full Grounded RAG Response

In [ ]:
# Final output format — what you'd show the user

def format_response(answer, docs):
    """Format the final answer with a clickable sources section."""
    doc_map = {d['id']: d for d in docs}
    cited_ids = extract_citations(answer)

    # Clean answer (remove the 'Sources used:' line — we'll add our own)
    clean_answer = answer.split("\nSources used:")[0].strip()

    output = f"{clean_answer}\n\n"
    output += "---\n📚 Sources:\n"
    for cid in cited_ids:
        d = doc_map.get(cid)
        if d:
            page = f", p.{d['page']}" if d['page'] else ""
            output += f"  [{cid}] {d['source']}{page}\n"

    return output


print("Final formatted response:\n")
print("=" * 60)
print(format_response(answer, docs))

## Key Takeaways 🎯

- **Grounding = prompt instructions** — tell the LLM explicitly to only use the sources
- **Number your docs** — makes citation natural and parseable
- **Parse citations** with a simple regex `[\d+]`
- **Faithfulness check** — flag sentences without citations as potential hallucinations
- For production, use a library like **RAGAS** for automated faithfulness scoring

---
Next: `04_Handling_Edge_Cases.ipynb` — what to do when retrieval finds nothing useful.